## PRT564 Assessment 4 — Market Cycle Classification
### Preprocessing & Exploratory Data Analysis
**Group 15 — Sydney Campus**

Classify each city-quarter into Boom / Normal / Decline based on YoY growth.

In [ ]:
import os

BASE   = r"C:\Users\ranas\OneDrive\Desktop\Rana_Research_Workspace\Rana_Research_Workspace"
RAW    = os.path.join(BASE, "Raw_Dataset")
CLEAN  = os.path.join(BASE, "Clean_Dataset")
GRAPHS = os.path.join(BASE, "EDA_Graphs")

os.makedirs(CLEAN, exist_ok=True)
os.makedirs(GRAPHS, exist_ok=True)
for p, n in [(RAW,"Raw_Dataset"),(CLEAN,"Clean_Dataset"),(GRAPHS,"EDA_Graphs")]:
    print(n, "found" if os.path.exists(p) else "NOT FOUND")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.05)
print("libraries loaded")

In [ ]:
CITIES = ['Sydney','Melbourne','Brisbane','Adelaide','Perth','Hobart','Darwin','Canberra']
CITIES_W_AVG = CITIES + ['Weighted_Avg']
STATES = ['NSW','Vic','Qld','SA','WA','Tas','NT','ACT','Australia']
CITY_TO_STATE = {'Sydney':'NSW','Melbourne':'Vic','Brisbane':'Qld','Adelaide':'SA',
                  'Perth':'WA','Hobart':'Tas','Darwin':'NT','Canberra':'ACT'}

In [ ]:
def parse_abs(path, sheet='Data1'):
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    data = raw.iloc[10:].copy()
    data = data.rename(columns={data.columns[0]: 'quarter'})
    data['quarter'] = pd.to_datetime(data['quarter'], errors='coerce')
    data = data.dropna(subset=['quarter'])
    for c in data.columns[1:]:
        data[c] = pd.to_numeric(data[c], errors='coerce')
    return data.reset_index(drop=True)

In [ ]:
print("loading 5 ABS files...")
df601 = parse_abs(os.path.join(RAW, '641601.xlsx'))
df602 = parse_abs(os.path.join(RAW, '641602.xlsx'))
df603 = parse_abs(os.path.join(RAW, '641603.xlsx'))
df604 = parse_abs(os.path.join(RAW, '641604.xlsx'))
df606 = parse_abs(os.path.join(RAW, '641606.xlsx'))
print("done")

In [ ]:
rppi = df601.iloc[:, :10].copy()
rppi.columns = ['quarter'] + [f'RPPI_{c}' for c in CITIES_W_AVG]

ehpi = df602.iloc[:, :10].copy()
ehpi.columns = ['quarter'] + [f'EHPI_{c}' for c in CITIES_W_AVG]

adpi = df603.iloc[:, :10].copy()
adpi.columns = ['quarter'] + [f'ADPI_{c}' for c in CITIES_W_AVG]

median_hp = df604.iloc[:, :9].copy()
median_hp.columns = ['quarter'] + [f'MedianHP_{c}' for c in CITIES]
for c in CITIES:
    median_hp[f'MedianHP_{c}'] *= 1000

transfers = df604.iloc[:, [0] + list(range(31, 39))].copy()
transfers.columns = ['quarter'] + [f'Transfers_{c}' for c in CITIES]

mean_dp = df606.iloc[:, [0] + list(range(27, 36))].copy()
mean_dp.columns = ['quarter'] + [f'MeanDP_{s}' for s in STATES]
for s in STATES:
    mean_dp[f'MeanDP_{s}'] *= 1000

num_dw = df606.iloc[:, [0] + list(range(36, 45))].copy()
num_dw.columns = ['quarter'] + [f'NumDw_{s}' for s in STATES]
for s in STATES:
    num_dw[f'NumDw_{s}'] *= 1000

hetero = mean_dp.merge(num_dw, on='quarter', how='inner')
print("columns extracted and renamed")

In [ ]:
base = (rppi
        .merge(ehpi,      on='quarter', how='outer')
        .merge(adpi,      on='quarter', how='outer')
        .merge(median_hp, on='quarter', how='left')
        .merge(transfers, on='quarter', how='left')
        .merge(hetero,    on='quarter', how='left')
        .sort_values('quarter').reset_index(drop=True))

base = base[base['RPPI_Sydney'].notna()].reset_index(drop=True)

hetero_cols = [c for c in base.columns if c.startswith('MeanDP_') or c.startswith('NumDw_')]
med_cols    = [c for c in base.columns if c.startswith('MedianHP_')]
trans_cols  = [c for c in base.columns if c.startswith('Transfers_')]

for col in hetero_cols:
    base[col] = pd.to_numeric(base[col], errors='coerce').interpolate(method='linear', limit_direction='both')
for col in med_cols + trans_cols:
    base[col] = base[col].ffill().bfill()

print(f"merged: {len(base)} rows x {base.shape[1]} columns")
print(f"missing values: {base.isna().sum().sum()}")

## Convert to Long Format and Create Classification Target

In [ ]:
long_rows = []
for _, row in base.iterrows():
    for city in CITIES:
        state = CITY_TO_STATE[city]
        long_rows.append({
            'quarter': row['quarter'], 'city': city, 'state': state,
            'RPPI': row[f'RPPI_{city}'],
            'EHPI': row[f'EHPI_{city}'],
            'ADPI': row[f'ADPI_{city}'],
            'MedianHP': row[f'MedianHP_{city}'],
            'Transfers': row[f'Transfers_{city}'],
            'MeanDP_state': row[f'MeanDP_{state}'],
            'NumDw_state':  row[f'NumDw_{state}'],
        })

df = pd.DataFrame(long_rows)
print(f"long format: {len(df)} rows ({len(base)} quarters x 8 cities)")

In [ ]:
df = df.sort_values(['city','quarter']).reset_index(drop=True)

def engineer(g):
    g['YoY_growth']    = g['RPPI'].pct_change(4) * 100
    g['QoQ_growth']    = g['RPPI'].pct_change(1) * 100
    g['RPPI_lag1']     = g['RPPI'].shift(1)
    g['RPPI_lag4']     = g['RPPI'].shift(4)
    g['EHPI_YoY']      = g['EHPI'].pct_change(4) * 100
    g['ADPI_YoY']      = g['ADPI'].pct_change(4) * 100
    g['MedianHP_YoY']  = g['MedianHP'].pct_change(4) * 100
    g['Transfers_YoY'] = g['Transfers'].pct_change(4) * 100
    g['log_RPPI']      = np.log(g['RPPI'])
    g['ma_4q']         = g['RPPI'].rolling(window=4, min_periods=1).mean()
    g['vol_4q']        = g['RPPI'].rolling(window=4, min_periods=2).std()
    g['price_per_dw']  = g['MeanDP_state'] / 1e6
    g['transfer_to_stock'] = g['Transfers'] / (g['NumDw_state'] + 1)
    return g

df = df.groupby('city', group_keys=False).apply(engineer)
print(f"feature engineering done - {df.shape[1]} columns")

In [ ]:
def label_cycle(yoy):
    if pd.isna(yoy):    return None
    if yoy > 10:        return 'Boom'
    elif yoy < 0:       return 'Decline'
    else:               return 'Normal'

df['cycle'] = df['YoY_growth'].apply(label_cycle)
df = df.dropna(subset=['cycle']).reset_index(drop=True)

print("class distribution:")
print(df['cycle'].value_counts())
print("\npercentages:")
print((df['cycle'].value_counts(normalize=True)*100).round(2))

In [ ]:
le_city = LabelEncoder()
df['city_encoded'] = le_city.fit_transform(df['city'])
df['quarter_num'] = pd.to_datetime(df['quarter']).dt.quarter
df['year'] = pd.to_datetime(df['quarter']).dt.year
df['time_idx'] = (df['year'] - 2003) * 4 + df['quarter_num']

features_to_scale = ['RPPI','EHPI','ADPI','MedianHP','Transfers','MeanDP_state',
                     'NumDw_state','RPPI_lag1','RPPI_lag4','EHPI_YoY','ADPI_YoY',
                     'MedianHP_YoY','Transfers_YoY','log_RPPI','ma_4q','vol_4q',
                     'price_per_dw','transfer_to_stock','QoQ_growth']

df_clean = df.dropna(subset=features_to_scale + ['cycle']).reset_index(drop=True)

scaler = StandardScaler()
df_clean[[f'{c}_scaled' for c in features_to_scale]] = scaler.fit_transform(df_clean[features_to_scale])

print(f"final dataset: {df_clean.shape[0]} rows x {df_clean.shape[1]} cols")
df_clean.to_csv(os.path.join(CLEAN, 'classification_dataset.csv'), index=False)
print("saved classification_dataset.csv")

## EDA — Visualisations for Classification

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))
ax.set_xlim(0, 10); ax.set_ylim(0, 13); ax.axis('off')
fig.patch.set_facecolor('#F7F9FC'); ax.set_facecolor('#F7F9FC')

boxes = [
    (5, 11.5, 'Raw ABS Data\n5 files: 641601 to 641606', '#2E6DA4', 'white'),
    (5,  9.5, 'Parse, Merge & Impute Missing Values', '#8E44AD', 'white'),
    (5,  7.5, 'Wide -> Long Format\n592 rows (74 quarters x 8 cities)', '#C0392B', 'white'),
    (5,  5.5, 'Feature Engineering\nYoY, QoQ, Lags, MA, Volatility', '#D68910', '#111'),
    (5,  3.5, 'Create Target Label\nBoom (>10%) | Normal (0-10%) | Decline (<0%)', '#1A5276', 'white'),
    (5,  1.5, 'Train: Naive Bayes + SVM + Random Forest', '#117A65', 'white'),
]
for (x, y, txt, fc, tc) in boxes:
    ax.add_patch(mpatches.FancyBboxPatch((x-3.8, y-0.7), 7.6, 1.4,
        boxstyle='round,pad=0.15', facecolor=fc, edgecolor='white', lw=2, zorder=3))
    ax.text(x, y, txt, ha='center', va='center', fontsize=9.5, color=tc, fontweight='bold', zorder=4)
for i in range(len(boxes)-1):
    ax.annotate('', xy=(5, boxes[i+1][1]+0.72), xytext=(5, boxes[i][1]-0.72),
        arrowprops=dict(arrowstyle='->', color='#444', lw=2))

ax.set_title('A4 - Classification Pipeline - Group 15', fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig(os.path.join(GRAPHS, 'A4_fig0_pipeline.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
order = ['Decline','Normal','Boom']
colors = ['#E74C3C','#F39C12','#27AE60']

counts = df_clean['cycle'].value_counts().reindex(order)
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=2)
for i, v in enumerate(counts.values):
    axes[0].text(i, v+5, str(v), ha='center', fontweight='bold')
axes[0].set_title('Class Distribution (Counts)', fontweight='bold')
axes[0].set_ylabel('Number of observations')

pct = df_clean['cycle'].value_counts(normalize=True).reindex(order) * 100
axes[1].pie(pct.values, labels=[f'{l}\n{v:.1f}%' for l,v in zip(pct.index, pct.values)],
            colors=colors, startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Class Distribution (Percentage)', fontweight='bold')

fig.suptitle('A4 Fig 1 - Market Cycle Class Distribution', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(GRAPHS, 'A4_fig1_class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
ct = pd.crosstab(df_clean['city'], df_clean['cycle'])[order]
ct.plot(kind='bar', stacked=True, ax=ax, color=colors, edgecolor='white', linewidth=1.5)
ax.set_title('A4 Fig 2 - Cycle Phase per City (stacked)', fontweight='bold')
ax.set_xlabel('City'); ax.set_ylabel('Number of quarters')
ax.legend(title='Cycle Class', loc='upper right')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(GRAPHS, 'A4_fig2_cycle_by_city.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 14), sharex=True)
axes = axes.flatten()

for i, city in enumerate(CITIES):
    sub = df_clean[df_clean['city']==city].copy()
    colors_pts = sub['cycle'].map({'Decline':'#E74C3C','Normal':'#F39C12','Boom':'#27AE60'})
    axes[i].scatter(sub['quarter'], sub['YoY_growth'], c=colors_pts, s=20, edgecolor='white', linewidth=0.5)
    axes[i].axhline(0, color='black', lw=0.8, ls='--', alpha=0.5)
    axes[i].axhline(10, color='green', lw=0.8, ls='--', alpha=0.4)
    axes[i].set_title(city, fontweight='bold')
    axes[i].set_ylabel('YoY %')

fig.suptitle('A4 Fig 3 - YoY Growth & Cycle Class by City', fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(GRAPHS, 'A4_fig3_cycle_over_time.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
feat_cols = ['RPPI_lag1','RPPI_lag4','EHPI_YoY','ADPI_YoY','MedianHP_YoY',
             'Transfers_YoY','ma_4q','vol_4q','price_per_dw','transfer_to_stock',
             'QoQ_growth','time_idx']
short = ['RPPI_L1','RPPI_L4','EHPI_Y','ADPI_Y','Med_Y','Tr_Y','MA4','Vol4','PpDw','TrStk','QoQ','Time']

corr_data = df_clean[feat_cols].dropna().astype(float).copy()
corr_data.columns = short
mask = np.triu(np.ones_like(corr_data.corr(), dtype=bool))

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_data.corr(), mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax, annot_kws={'size':9})
ax.set_title('A4 Fig 4 - Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(GRAPHS, 'A4_fig4_correlation.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
features = ['RPPI_lag1','EHPI_YoY','MedianHP_YoY','Transfers_YoY','vol_4q','QoQ_growth']
titles   = ['RPPI Lag-1','EHPI YoY','MedianHP YoY','Transfers YoY','Volatility 4Q','QoQ Growth']
class_colors = {'Decline':'#E74C3C','Normal':'#F39C12','Boom':'#27AE60'}

for i, (col, title) in enumerate(zip(features, titles)):
    ax = axes[i//3, i%3]
    for c in ['Decline','Normal','Boom']:
        sub = df_clean[df_clean['cycle']==c][col].dropna()
        ax.hist(sub, bins=20, alpha=0.55, label=c, color=class_colors[c], edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=8)
fig.suptitle('A4 Fig 5 - Feature Distributions by Cycle Class', fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(GRAPHS, 'A4_fig5_feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
features = ['YoY_growth','MedianHP_YoY','EHPI_YoY','vol_4q']
titles   = ['YoY Growth %','MedianHP YoY %','EHPI YoY %','Volatility (4Q std)']

for i, (col, title) in enumerate(zip(features, titles)):
    ax = axes[i//2, i%2]
    sns.boxplot(data=df_clean, x='cycle', y=col, order=order, ax=ax,
                palette=['#E74C3C','#F39C12','#27AE60'])
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(''); ax.set_ylabel(col)
fig.suptitle('A4 Fig 6 - Feature Boxplots by Cycle Class', fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(GRAPHS, 'A4_fig6_boxplots.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
summary = df_clean.groupby('cycle')[['YoY_growth','MedianHP_YoY','EHPI_YoY','vol_4q','Transfers_YoY']].mean().round(3)
print("Mean feature values per class:")
print(summary)
summary.to_csv(os.path.join(CLEAN, 'class_summary_stats.csv'))
print(f"\nfinal dataset shape: {df_clean.shape}")
print("EDA complete - all figures saved to EDA_Graphs/")